In [1]:
import copy
import pandas as pd
import numpy as np
import plotly.io
from scipy.optimize import minimize
import TradeAndLogics as TL

In [2]:
import Data as data
import Data_Processing as dp
from enums import *

In [3]:
import importlib

In [4]:
def f(constituents, constituent_symbol, weight, lot_size_futures, lot_size_options):
    dict = {
        'weight': weight,
        'lot_size_futures': lot_size_futures,
        'lot_size_options': lot_size_options,    
    }
    constituents[constituent_symbol] = dict

In [5]:
constituents_weight_lots = {}

In [6]:
index_symbol = 'BANKNIFTY'
index_options_lot_size = 15
index_futures_lot_size = 15

In [7]:
# f(constituents_weight_lots, "PNB", 0.91, 8000, 8000)
# f(constituents_weight_lots, "ICICIBANK", 23.03, 700, 700)
f(constituents_weight_lots, "ICICIBANK", 50, 700, 700)
# f(constituents_weight_lots, "AUBANK", 2.69, 1000, 1000)
# f(constituents_weight_lots, "BANKBARODA", 1.84, 2925, 2925)
# f(constituents_weight_lots, "FEDERALBNK", 1.68, 5000, 5000)
# f(constituents_weight_lots, "IDFCFIRSTB", 1.08, 7500, 7500)
# f(constituents_weight_lots, "SBIN", 11.27, 750, 750)
# f(constituents_weight_lots, "INDUSINDBK", 5.58, 500, 500)
# f(constituents_weight_lots, "HDFCBANK", 27.04, 550, 550)
f(constituents_weight_lots, "HDFCBANK", 50, 550, 550)
# f(constituents_weight_lots, "KOTAKBANK", 11.72, 400, 400)
# f(constituents_weight_lots, "BANDHANBNK", 1.98, 2800, 2800)
# f(constituents_weight_lots, "AXISBANK", 11.18, 625, 625)

In [8]:
constituents_weight_lots

{'ICICIBANK': {'weight': 50, 'lot_size_futures': 700, 'lot_size_options': 700},
 'HDFCBANK': {'weight': 50, 'lot_size_futures': 550, 'lot_size_options': 550}}

In [9]:
sum = 0
for values in constituents_weight_lots.values():
    sum += values['weight']
print(sum)

100


In [10]:
start_date = '2023-07-01'
end_date = '2023-12-01'
expiry_type = 'I'
r = 0.1 # (10% interest rate)
t=15
look_back_window = 5*25*4

In [11]:
importlib.reload(dp)
importlib.reload(data)

<module 'Data' from 'C:\\Users\\vinayak\\Desktop\\Backtesting\\Data.py'>

In [12]:
# cursor, conn = data.make_connection_to_db(False)

In [13]:
# cursor.execute(
#         f'''
#                 SELECT *
#                 FROM ohlcv_future_per_1_minute ofpm
#                 WHERE ofpm.symbol = 'PNB'
#                 AND DATE(ofpm.date_timestamp) = '2023-12-05'
#                 AND ofpm.expiry_type = 'I'
#                 ORDER BY date_timestamp ASC;
#             '''
#     )
# rows = cursor.fetchall()
# pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description])

In [14]:
# cursor.close()
# conn.close()

In [12]:
constituents_weights_lots_data = {}

In [13]:
for constituent_symbol, constituent in constituents_weight_lots.items():
    df_futures = data.get_futures_data_with_timestamps_between(constituent_symbol, expiry_type, start_date, end_date, t)
    df_options = data.get_options_data_with_timestamps_between(constituent_symbol, expiry_type, start_date, end_date, t)
    constituents_weights_lots_data[constituent_symbol] = constituents_weight_lots[constituent_symbol]
    constituents_weights_lots_data[constituent_symbol]['df_options'] = df_options
    constituents_weights_lots_data[constituent_symbol]['df_futures'] = df_futures

Fetching ICICIBANK's-I Futures Data, Timeframe = 15mins, start : 2023-07-01, end : 2023-12-01
-----------------------------------------------------------------
2023-07-03 00:00:00
-----------------------------------------------------------------
2023-07-04 00:00:00
-----------------------------------------------------------------
2023-07-05 00:00:00
-----------------------------------------------------------------
2023-07-06 00:00:00
-----------------------------------------------------------------
2023-07-07 00:00:00
-----------------------------------------------------------------
2023-07-10 00:00:00
-----------------------------------------------------------------
2023-07-11 00:00:00
-----------------------------------------------------------------
2023-07-12 00:00:00
-----------------------------------------------------------------
2023-07-13 00:00:00
-----------------------------------------------------------------
2023-07-14 00:00:00
----------------------------------------------

In [14]:
df_futures_index = data.get_futures_data_with_timestamps_between(index_symbol, expiry_type, start_date, end_date, t)
df_options_index = data.get_options_data_with_timestamps_between(index_symbol, expiry_type, start_date, end_date, t)

Fetching BANKNIFTY's-I Futures Data, Timeframe = 15mins, start : 2023-07-01, end : 2023-12-01
-----------------------------------------------------------------
2023-07-03 00:00:00
-----------------------------------------------------------------
2023-07-04 00:00:00
-----------------------------------------------------------------
2023-07-05 00:00:00
-----------------------------------------------------------------
2023-07-06 00:00:00
-----------------------------------------------------------------
2023-07-07 00:00:00
-----------------------------------------------------------------
2023-07-10 00:00:00
-----------------------------------------------------------------
2023-07-11 00:00:00
-----------------------------------------------------------------
2023-07-12 00:00:00
-----------------------------------------------------------------
2023-07-13 00:00:00
-----------------------------------------------------------------
2023-07-14 00:00:00
----------------------------------------------

In [15]:
importlib.reload(dp)
importlib.reload(data)

<module 'Data' from 'C:\\Users\\vinayak\\Desktop\\Backtesting\\Data.py'>

In [63]:
constituents = {}

In [64]:
for constituent_symbol, constituent in constituents_weights_lots_data.items():
    constituents[constituent_symbol] = dp.ticker(constituent_symbol, constituent['df_futures'], constituent['df_options'], False, None, True, constituent['weight'], constituent['lot_size_options'], constituent['lot_size_futures'], t, look_back_window, False)

In [65]:
index = dp.ticker(index_symbol, df_futures_index, df_options_index, True, constituents, False, None, index_options_lot_size, index_futures_lot_size, t, look_back_window)

In [66]:
ohlc = OHLC.close

In [67]:
index.set_ohlc(ohlc)
for _, constituent in constituents.items():
    constituent.set_ohlc(ohlc)

In [68]:
def get_index_and_components_iv(index):
    index.generate_iv_data()
    for _, component in index.components.items():
        component.generate_iv_data()

In [69]:
get_index_and_components_iv(index)

In [70]:
index.generate_ic_data()

In [71]:
index.df_futures['zscore'] = (index.df_futures['ic'] - index.df_futures['ic'].rolling(window=index.look_back_window).mean())/index.df_futures['ic'].rolling(window=index.look_back_window).std()

In [26]:
# index.df_futures['iv'].unique()

In [27]:
# index.df_futures['ic'].isna().sum()


In [28]:
# index.df_futures['zscore'].unique()

In [29]:
# index.df_futures['ic'].describe()

In [30]:
# index.df_futures['zscore'].describe()

In [31]:
# index.df_futures['zscore'][index.df_futures['zscore'] > 1.5]

In [32]:
# index.df_futures['ix'] = range(1, len(index.df_futures) + 1)

In [33]:
# index.df_futures['zscore']

In [72]:
import Plot
importlib.reload(Plot)

<module 'Plot' from 'C:\\Users\\vinayak\\Desktop\\Backtesting\\Plot.py'>

In [73]:
fig_ic = Plot.plot_df(index.df_futures, 'ic')
fig_ic.show()

In [74]:
fig_zscore = Plot.plot_df(index.df_futures, 'zscore')
fig_zscore = Plot.draw_horizontal_line(fig_zscore, -2, 0, len(index.df_futures), 'Red')
fig_zscore = Plot.draw_horizontal_line(fig_zscore, -1.5, 0, len(index.df_futures), 'Blue')
fig_zscore = Plot.draw_horizontal_line(fig_zscore, 1.5, 0, len(index.df_futures), 'Blue')
fig_zscore = Plot.draw_horizontal_line(fig_zscore, 2, 0, len(index.df_futures), 'Red')
fig_zscore.show()


In [75]:
positions = {}

In [76]:
dispersion_position = LongShort.Long

In [77]:
def entry_signal(timestamp):
    zscore = index.df_futures.loc[timestamp, 'zscore']
    return zscore > 1.5


def exit_signal(timestamp):
    zscore = index.df_futures.loc[timestamp, 'zscore'] 
    return zscore < -1.5

def get_lots_for_dispersion(dispersion_position_of_index, index, timestamp, delta_neutral = False, vega_neutral = False, theta_neutral = False):
    # If you want to do vega neutral entry
    if vega_neutral: 
        try:
            vegas = []
            vega_index_atm_straddle = get_straddle_vega(index, timestamp) * dispersion_position_of_index.value
            if vega_index_atm_straddle is None:
                return None, None
            vegas.append(vega_index_atm_straddle)
            for _, component in index.components.items():
                vega_component_atm_straddle = get_straddle_vega(component, timestamp) * dispersion_position_of_index.value * -1
                if vega_component_atm_straddle is None:
                    return None, None
                vegas.append(vega_component_atm_straddle)
            return get_lots_for_neutrality(vegas)
        except Exception as e:
            print(f"get_lots_for_dispersion error at {timestamp}: {e}")
    # If you want to do a delta neutral entry
    if delta_neutral:
        pass
    # If you want to do a theta/ correlation neutral entry
    if theta_neutral:
        pass

In [78]:
def get_straddle_vega(ticker, timestamp):
    vega = 0
    for option_type in Option:
        strike, _ = ticker.find_moneyness_strike(timestamp, 0, option_type)
        greeks = ticker.greeks(timestamp, option_type, strike)
        if greeks is None:
            return None
        vega += greeks['vega']
    return vega

In [79]:
def get_net_delta(ticker, timestamp):
    net_delta = 0
    for _, token in ticker.tokens.items():
        net_delta += token.stats.loc[timestamp, 'net_delta'] * ticker.lot_size_options
    # net_delta += ticker.net_futures_delta
    return net_delta

In [80]:
def objective(x, v):
    #objective function: net vega = sum(x_i * v_i) + X * V
    return abs(np.dot(x, v))

def get_lots_for_neutrality(v):
    print(f"vegas (index, *components): {v}")
    n = len(v)
    x0 = np.ones(n)
    bounds = [(0, None)] * (n)
    result = minimize(objective, x0, args=(v), method='SLSQP', bounds=bounds)
    x_rounded = np.round(result.x).astype(int)
    return x_rounded[0], x_rounded[1:]

In [81]:
def take_dispersion_position(time_data, ticker, position, lots):
    timestamp, time_ix = time_data
    # print(f"timestamp, time_ix: {timestamp, time_ix}")
    legs = []
    for option_type in Option:
        atm_strike, _ = ticker.find_moneyness_strike(timestamp, 0, option_type)
        # print(f"atm_strike: {atm_strike}")
        key = f'{ticker.symbol}_{option_type.name}_{atm_strike}' 
        if key not in positions:
            positions[key] = TL.Token(ticker, option_type, atm_strike)
        token = positions[key]
        token.add_position(lots, position)
        token.update_df(time_ix, timestamp)
        ticker.update_token(key, token)
        leg = {
            "position" : position, 
            "opt_type" : option_type, 
            "moneyness": 0, 
            "lots": lots
        }
        # print(leg)
        legs.append(leg)
    ticker.take_position(timestamp, *legs)

In [82]:
def check_existing_position(ticker):
    pos = False
    for _, token in ticker.tokens.items():
        pos = pos | (token.position.value != 0) 
    return pos

In [83]:
def round(x):
    return np.round(x, 2)

In [84]:
def hedge_delta(time_ix, timestamp, ticker):
    net_delta = get_net_delta(ticker, timestamp)
    if abs(net_delta) >= ticker.lot_size_futures:
        qty_futures = (-net_delta)/ticker.lot_size_futures
        position = LongShort(np.sign(qty_futures))
        lots_futures = int(abs(qty_futures))
        ticker.hedge_futures_trade(lots_futures, position, timestamp)  
        key = f'{ticker.symbol}_Futures'
        if key not in positions:
            positions[key] = TL.Token(ticker, None, None, FNO.FUTURES)
        token = positions[key]
        token.add_position(lots_futures, position)
        token.update_df(time_ix, timestamp)
        ticker.update_token(key, token)
        token.ticker = ticker
        positions[key] = token
        print(f">> Hedged delta: Added {lots_futures * position.value * ticker.lot_size_futures} notional delta using {lots_futures} lots of future. new delta = {round(net_delta) + lots_futures * position.value * ticker.lot_size_futures}")
    else:
        print(f">> Did not hedge {round(net_delta)} notional delta for {ticker.symbol} at {timestamp}. net_delta/{ticker.symbol}_lot_size_futures = {np.round(abs(net_delta)/ticker.lot_size_futures)} < 1")

In [85]:
def reset_hedging_positions(ticker):
    ticker.hedging = pd.DataFrame(columns = ['date_timestamp', 'lots', 'position', 'futures_price', 'delta_added'])
    ticker.hedging.set_index('date_timestamp', inplace=True)

def reset_trade_positions(ticker):
    ticker.Trades = dp.Trades()
    ticker.tokens = {}

In [86]:
def update(ticker, time_ix, timestamp):
    for key, token in ticker.tokens.items():
        token.update_df(time_ix, timestamp)
        positions[key].update_df(time_ix, timestamp)
        ticker.update_token(key, token)
    for key, token in ticker.tokens.items():
        token.ticker = ticker
        positions[key] = token

In [87]:
def update_and_hedge(ticker, time_ix, timestamp, print=True):
    for key, token in ticker.tokens.items():
        token.update_df(time_ix, timestamp)
        positions[key] = token
        # ticker.update_token(key, token)
        if not print:
            continue
        if token.fno == FNO.OPTIONS:
            print(f"{token.ticker.symbol, token.opt_type.name, float(token.strike/100)} | Delta per lot = {np.round(token.stats.loc[timestamp, 'net_delta'], 2)} (position accounted) | Notional Delta = {np.round(token.stats.loc[timestamp, 'net_delta'], 2) * ticker.lot_size_options} | Lot size: {ticker.lot_size_options} | Lots in position: {token.lots} | position = {token.position.name}")
        else:
            print(f"{token.ticker.symbol, token.fno.name} | Delta per lot = {np.round(token.stats.loc[timestamp, 'net_delta'], 2)} | Notional Delta = {np.round(token.stats.loc[timestamp, 'net_delta'], 2) * ticker.lot_size_options} | Lot size: {ticker.lot_size_futures} | Lots in position: {token.lots} | position = {token.position.name}")
    # if abs(ticker.hedging['lots'].sum()) > 0:
    #     print(f"{ticker.symbol} Futures | Delta per lot = {ticker.net_futures_delta/abs(ticker.hedging['lots'].sum())} | Notional Delta = {ticker.net_futures_delta}")
    hedge_delta(time_ix, timestamp, ticker)
    for key, token in ticker.tokens.items():
        token.ticker = ticker
        positions[key] = token

In [88]:
importlib.reload(TL)
importlib.reload(data)
importlib.reload(dp)

<module 'Data_Processing' from 'C:\\Users\\vinayak\\Desktop\\Backtesting\\Data_Processing.py'>

In [89]:
# index.components['BANDHANBNK'].hedging

In [90]:
def get_running_pnl(timestamp):
    net_profit = 0
    for token in positions.values():
        net_profit += token.stats.loc[timestamp, 'running_pnl']
    return net_profit

In [91]:
def ptsl(timestamp, running_pnl_at_last_entry, pt, sl):
    net_profit = get_running_pnl(timestamp) - running_pnl_at_last_entry
    if net_profit >= pt:
        return True
    if net_profit < sl:
        return True
    return False

In [92]:
index.tokens

{}

In [93]:
# positions['HDFCBANK_Put_164000'].stats.to_excel("HDFCBANK_Put_164000_positions.xlsx")

In [94]:
# x = list(index.tokens.values())[1]
# x.stats.to_excel("bnf_call_46000.xlsx")

In [95]:
def squareoff(time_ix, timestamp):
    for key, token in positions.items():
        token.add_position(token.lots, LongShort(-token.position.value))
        token.update_df(time_ix, timestamp)
        positions[key] = token
    for key, token in index.tokens.items():
        token.add_position(token.lots, LongShort(-token.position.value))
        token.update_df(time_ix, timestamp)
        positions[key] = token
        index.tokens[key] = token
    for key, token in index.tokens.items():
        token.ticker = index
        positions[key].ticker = index
    for component in index.components.values():
        for key, token in component.tokens.items():
            token.add_position(token.lots, LongShort(-token.position.value))
            token.update_df(time_ix, timestamp)
            positions[key] = token
            component.tokens[key] = token
    for component in index.components.values():
        for key, token in component.tokens.items():
            token.ticker = component
            positions[key].ticker = component
    update(index, time_ix, timestamp)
    for component in index.components.values():
        update(component, time_ix, timestamp)

In [ ]:
positions = {}
import sys
reset_hedging_positions(index)
reset_trade_positions(index)
for component in index.components.values():
    reset_hedging_positions(component)
    reset_trade_positions(component)
# print(pd.Series(index.df_futures.index.date).unique())

df = pd.DataFrame("No", index = index.df_futures.index, columns=["PTSL", "ENTRY", "EXIT", "GreeksWereNotAvailable", "Existing Position"])
flag = True
running_pnl_at_last_entry = 0
with open('output_logs_new.txt', 'w') as f:
    sys.stdout = f
    
    for time_ix, timestamp in enumerate(index.df_futures.index):
        # print(f"time_ix, timestamp, signal: {time_ix, timestamp, entry_signal(timestamp)}")
        print()
        print(f"{timestamp}")
        if check_existing_position(index):
            
            df.loc[timestamp, "Existing Position"] = "Yes"
            print("Existing Position check: TRUE")
            if ptsl(timestamp, running_pnl_at_last_entry, 1000000, -1000000):
                print("************  PTSL Trigger hit *********** ")
                df.loc[timestamp, "PTSL"] = "Yes"
                squareoff(time_ix, timestamp)
                update_and_hedge(index, time_ix, timestamp, False)
                for component in index.components.values():
                    update_and_hedge(component, time_ix, timestamp, False)
                continue
            
            squareoff_signal = exit_signal(timestamp) if flag else entry_signal(timestamp)
            if squareoff_signal:
                df.loc[timestamp, "EXIT"] = "Yes" if flag else "No"
                df.loc[timestamp, "ENTRY"] = "No" if flag else "Yes"
                print("************  SquareOff Reverse zscore Signal hit. ************ ")
                squareoff(time_ix, timestamp)
                update_and_hedge(index, time_ix, timestamp, False)
                for component in index.components.values():
                    update_and_hedge(component, time_ix, timestamp, False)
                continue
                 
            is_expiry = False
            is_expiry |= pd.to_datetime(timestamp).date() == pd.to_datetime(index.get_opts(Option.Call).loc[timestamp, 'expiry']).date()
            for component in index.components.values():
                is_expiry |= pd.to_datetime(timestamp).date() == pd.to_datetime(component.get_opts(Option.Call).loc[timestamp, 'expiry']).date()
            if is_expiry:
                squareoff(time_ix, timestamp)
                update_and_hedge(index, time_ix, timestamp, False)
                for component in index.components.values():
                    update_and_hedge(component, time_ix, timestamp, False)
                print("************  EXPIRY DAY, MANUAL SQUARE OFF ************  ")
                continue
             
            update_and_hedge(index, time_ix, timestamp)
            for component in index.components.values():
                update_and_hedge(component, time_ix, timestamp)
            print("================")
            continue
        
        if entry_signal(timestamp):
            print("************  ENTRY SIGNAL ************  ")
            df.loc[timestamp, "ENTRY"] = "Yes"
            try:
                lots_index, lots_components = get_lots_for_dispersion(LongShort.Short, index, timestamp, False, True, False)
                print(f"lots_index, lots_components: {lots_index, lots_components}")
    
                take_dispersion_position((timestamp, time_ix), index, LongShort.Short, lots_index)
                for (component, lots_component) in zip(index.components.values(), lots_components):
                    take_dispersion_position((timestamp, time_ix), component, LongShort.Long, lots_component)
                flag = True
                print(f"Short dispersion (Short Index Long Components) trade at {timestamp} when zscore was {index.df_futures.loc[timestamp, 'zscore']}")
                running_pnl_at_last_entry = get_running_pnl(timestamp)
            except Exception as e:
                print(f"Can't take trade due to Non availability of greeks -> cant enter a vega neutral position")
                df.loc[timestamp, "GreeksWereNotAvailable"] = "Yes"
                print(f"take_dispersion_position error at {timestamp}: {e}")  
            # continue
                
    
            
        if exit_signal(timestamp):
            print("************  EXIT SIGNAL ************  ")
            df.loc[timestamp, "EXIT"] = "Yes"
            try:
                lots_index, lots_components = get_lots_for_dispersion(LongShort.Long, index, timestamp, False, True, False)
                print(f"lots_index, lots_components: {lots_index, lots_components}")
    
                take_dispersion_position((timestamp, time_ix), index, LongShort.Long, lots_index)
                for (component, lots_component) in zip(index.components.values(), lots_components):
                    take_dispersion_position((timestamp, time_ix), component, LongShort.Short, lots_component)
                flag = False
                print(f"Long dispersion (Long Index Short Components) trade at {timestamp} when zscore was {index.df_futures.loc[timestamp, 'zscore']}")
                running_pnl_at_last_entry = get_running_pnl(timestamp)
            except Exception as e:
                print(f"Can't take trade due to Non availability of greeks -> cant enter a vega neutral position")
                df.loc[timestamp, "GreeksWereNotAvailable"] = "Yes"
                print(f"take_dispersion_position error at {timestamp}: {e}") 
            # continue
        
        update_and_hedge(index, time_ix, timestamp, False)
        for component in index.components.values():
            update_and_hedge(component, time_ix, timestamp, False)
        
sys.stdout = sys.__stdout__

In [97]:
print(2)

In [96]:
sys.stdout = sys.__stdout__
# from IPython import display
# display.clear_output(wait=True)

In [70]:
df.to_excel("trade_execution_icici_hdfc_bnf.xlsx")

In [71]:
importlib.reload(Plot)

<module 'Plot' from 'C:\\Users\\vinayak\\Desktop\\Backtesting\\Plot.py'>

In [72]:
Plot.save_excel_trade_execution("trade_execution_icici_hdfc_bnf.xlsx")

In [139]:
# list(positions.values())[0].stats

In [140]:
positions.keys()

dict_keys(['BANKNIFTY_Put_4600000', 'BANKNIFTY_Call_4600000', 'ICICIBANK_Put_99000', 'ICICIBANK_Call_99000', 'HDFCBANK_Put_164000', 'HDFCBANK_Call_164000', 'HDFCBANK_Futures', 'ICICIBANK_Futures', 'BANKNIFTY_Put_4460000', 'BANKNIFTY_Call_4460000', 'ICICIBANK_Put_95000', 'ICICIBANK_Call_95000', 'HDFCBANK_Put_153000', 'HDFCBANK_Call_153000', 'BANKNIFTY_Put_4300000', 'BANKNIFTY_Call_4300000', 'ICICIBANK_Put_92000', 'ICICIBANK_Call_92000', 'HDFCBANK_Put_148000', 'HDFCBANK_Call_148000'])

In [ ]:
with open("whathappenedat1215.txt", 'w') as f:
    sys.stdout = f
    for key, token in positions.items():
        print(key)
        print(token.stats.loc['2023-09-25 12:15:00'])
        print()
    sys.stdout.flush()

In [141]:
# ps = []
# for timestamp in index.df_futures.index:
#     p = 0
#     for _, token in positions.items():
#         p += token.stats.loc[timestamp, 'running_pnl']
#     ps.append(p)
# pd.Series(ps).unique()

In [142]:
index.Trades.tradesArr

[{'timestamp': Timestamp('2023-09-14 13:15:00'),
  'PriceLeg0': np.float64(39535.0),
  'StrikeLeg0': np.int64(4600000),
  'OptTypeLeg0': <Option.Put: 0>,
  'MoneynessLeg0': 0,
  'PositionLeg0': <LongShort.Long: 1>,
  'LotsLeg0': np.int64(1),
  'PriceLeg1': np.float64(44595.0),
  'StrikeLeg1': np.int64(4600000),
  'OptTypeLeg1': <Option.Call: 1>,
  'MoneynessLeg1': 0,
  'PositionLeg1': <LongShort.Long: 1>,
  'LotsLeg1': np.int64(1)},
 {'timestamp': Timestamp('2023-09-22 15:00:00'),
  'PriceLeg0': np.float64(25725.0),
  'StrikeLeg0': np.int64(4460000),
  'OptTypeLeg0': <Option.Put: 0>,
  'MoneynessLeg0': 0,
  'PositionLeg0': <LongShort.Long: 1>,
  'LotsLeg0': np.int64(1),
  'PriceLeg1': np.float64(28620.0),
  'StrikeLeg1': np.int64(4460000),
  'OptTypeLeg1': <Option.Call: 1>,
  'MoneynessLeg1': 0,
  'PositionLeg1': <LongShort.Long: 1>,
  'LotsLeg1': np.int64(1)},
 {'timestamp': Timestamp('2023-10-27 13:30:00'),
  'PriceLeg0': np.float64(65315.0),
  'StrikeLeg0': np.int64(4300000),
  'Opt

In [48]:
data = {
    'running_pnl': [],
    'net_delta': [],
    'net_vega': [],
    'net_lots': [],
}

for timestamp in index.df_futures.index:
    pnl = 0
    delta = 0
    vega = 0
    lots = 0

    for _, token in positions.items():
        stats_at_timestamp = token.stats.loc[timestamp]
        pnl += stats_at_timestamp['running_pnl']
        delta += stats_at_timestamp['net_delta']
        vega += stats_at_timestamp['net_vega']
        lots += stats_at_timestamp['net_lots']

    data['running_pnl'].append(pnl / 100) 
    data['net_delta'].append(delta)
    data['net_vega'].append(vega)
    data['net_lots'].append(lots)

result_df = pd.DataFrame(data, index=index.df_futures.index)

In [49]:
result_df['zscore'] = index.df_futures['zscore']*100

In [50]:
result_df['active_position'] = df['Existing Position'].map({"Yes": 1, "No": 0})
result_df['active_position'] *= 500

In [62]:
importlib.reload(Plot)

<module 'Plot' from 'C:\\Users\\vinayak\\Desktop\\Backtesting\\Plot.py'>

In [63]:
fig = Plot.plot_df(result_df, 'running_pnl', 'net_delta', 'net_vega', 'net_lots', 'zscore', 'active_position')
fig = Plot.draw_horizontal_line(fig, -150, index.df_futures['ix'].iloc[0], index.df_futures['ix'].iloc[-1], 'white')
fig = Plot.draw_horizontal_line(fig, 150, index.df_futures['ix'].iloc[0], index.df_futures['ix'].iloc[-1], 'white')
fig.show()
Plot.save_plot(fig, 'output_bnf_icici_hdfc.html')

In [53]:
result_df.index = pd.to_datetime(result_df.index)
result_df['date'] = result_df.index.date  # Create a new column for the date
daily_profit = result_df.groupby('date')['running_pnl'].agg(lambda x: x.iloc[-1] - x.iloc[0])

In [74]:
x = daily_profit.unique()
x
daily_profit[daily_profit > 0]

date
2023-09-14     13.20
2023-09-15     67.95
2023-09-20    206.30
2023-09-21    621.40
2023-09-22     67.90
2023-09-25      3.95
2023-10-31     75.70
2023-11-01     20.05
2023-11-02     92.65
2023-11-08     82.20
2023-11-13     15.65
2023-11-15     54.25
2023-11-16     51.50
2023-11-17    145.20
2023-11-21     93.55
2023-11-22     73.25
2023-11-23     33.10
Name: running_pnl, dtype: float64

In [75]:
x.sum()*100/1000000 * 100

np.float64(4.263999999999997)

In [76]:
x.mean()/x.std()

np.float64(0.09108939658216153)

In [77]:
# result_df['date'].unique()

In [78]:
arr = []
arr += [{**trade, 'symbol': index.symbol} for trade in index.Trades.tradesArr]
for comp in index.components.values():
    arr += [{**trade, 'symbol': comp.symbol} for trade in comp.Trades.tradesArr]
arr

[{'timestamp': Timestamp('2023-09-14 13:15:00'),
  'PriceLeg0': np.float64(39535.0),
  'StrikeLeg0': np.int64(4600000),
  'OptTypeLeg0': <Option.Put: 0>,
  'MoneynessLeg0': 0,
  'PositionLeg0': <LongShort.Long: 1>,
  'LotsLeg0': np.int64(1),
  'PriceLeg1': np.float64(44595.0),
  'StrikeLeg1': np.int64(4600000),
  'OptTypeLeg1': <Option.Call: 1>,
  'MoneynessLeg1': 0,
  'PositionLeg1': <LongShort.Long: 1>,
  'LotsLeg1': np.int64(1),
  'symbol': 'BANKNIFTY'},
 {'timestamp': Timestamp('2023-09-22 15:00:00'),
  'PriceLeg0': np.float64(25725.0),
  'StrikeLeg0': np.int64(4460000),
  'OptTypeLeg0': <Option.Put: 0>,
  'MoneynessLeg0': 0,
  'PositionLeg0': <LongShort.Long: 1>,
  'LotsLeg0': np.int64(1),
  'PriceLeg1': np.float64(28620.0),
  'StrikeLeg1': np.int64(4460000),
  'OptTypeLeg1': <Option.Call: 1>,
  'MoneynessLeg1': 0,
  'PositionLeg1': <LongShort.Long: 1>,
  'LotsLeg1': np.int64(1),
  'symbol': 'BANKNIFTY'},
 {'timestamp': Timestamp('2023-10-27 13:30:00'),
  'PriceLeg0': np.float64(6

In [79]:
df_trades = pd.DataFrame(arr)
df_trades = df_trades.sort_values(by='timestamp')
df_trades.reset_index(inplace=True)
df_trades.drop(columns=['index'], inplace=True)

In [80]:
df_trades

,timestamp,PriceLeg0,StrikeLeg0,OptTypeLeg0,MoneynessLeg0,PositionLeg0,LotsLeg0,PriceLeg1,StrikeLeg1,OptTypeLeg1,MoneynessLeg1,PositionLeg1,LotsLeg1,symbol
0,2023-09-14 13:15:00,39535.0,4600000,Option.Put,0,LongShort.Long,1,44595.0,4600000,Option.Call,0,LongShort.Long,1,BANKNIFTY
1,2023-09-14 13:15:00,1515.0,99000,Option.Put,0,LongShort.Short,12,1490.0,99000,Option.Call,0,LongShort.Short,12,ICICIBANK
2,2023-09-14 13:15:00,1760.0,164000,Option.Put,0,LongShort.Short,19,2165.0,164000,Option.Call,0,LongShort.Short,19,HDFCBANK
3,2023-09-22 15:00:00,25725.0,4460000,Option.Put,0,LongShort.Long,1,28620.0,4460000,Option.Call,0,LongShort.Long,1,BANKNIFTY
4,2023-09-22 15:00:00,860.0,95000,Option.Put,0,LongShort.Short,12,935.0,95000,Option.Call,0,LongShort.Short,12,ICICIBANK
5,2023-09-22 15:00:00,1390.0,153000,Option.Put,0,LongShort.Short,19,1830.0,153000,Option.Call,0,LongShort.Short,19,HDFCBANK
6,2023-10-27 13:30:00,65315.0,4300000,Option.Put,0,LongShort.Short,1,62500.0,4300000,Option.Call,0,LongShort.Short,1,BANKNIFTY
7,2023-10-27 13:30:00,1735.0,92000,Option.Put,0,LongShort.Long,13,1760.0,92000,Option.Call,0,LongShort.Long,13,ICICIBANK
8,2023-10-27 13:30:00,2960.0,148000,Option.Put,0,LongShort.Long,20,2940.0,148000,Option.Call,0,LongShort.Long,20,HDFCBANK


In [82]:
df_trades.to_excel("df_trades_icici_hdfc_bnf.xlsx", index=False)

In [79]:
positions

{'BANKNIFTY_Put_4600000': <TradeAndLogics.Token at 0x276bae4e210>,
 'BANKNIFTY_Call_4600000': <TradeAndLogics.Token at 0x276baeecce0>,
 'ICICIBANK_Put_99000': <TradeAndLogics.Token at 0x276bae4e2a0>,
 'ICICIBANK_Call_99000': <TradeAndLogics.Token at 0x276bbab76e0>,
 'HDFCBANK_Put_164000': <TradeAndLogics.Token at 0x276bbb10dd0>,
 'HDFCBANK_Call_164000': <TradeAndLogics.Token at 0x276bba8f590>,
 'HDFCBANK_Futures': <TradeAndLogics.Token at 0x276b8ce8890>,
 'ICICIBANK_Futures': <TradeAndLogics.Token at 0x276ae063da0>,
 'BANKNIFTY_Put_4460000': <TradeAndLogics.Token at 0x276fb1f37d0>,
 'BANKNIFTY_Call_4460000': <TradeAndLogics.Token at 0x276bace3da0>,
 'ICICIBANK_Put_95000': <TradeAndLogics.Token at 0x276baea1af0>,
 'ICICIBANK_Call_95000': <TradeAndLogics.Token at 0x276ac0f3b90>,
 'HDFCBANK_Put_153000': <TradeAndLogics.Token at 0x276bbab5d00>,
 'HDFCBANK_Call_153000': <TradeAndLogics.Token at 0x276b7dd08f0>,
 'BANKNIFTY_Put_4300000': <TradeAndLogics.Token at 0x276bbab7320>,
 'BANKNIFTY_Ca

In [83]:
Plot.save_excel_trades("df_trades_icici_hdfc_bnf.xlsx")

In [82]:
importlib.reload(Plot)

<module 'Plot' from 'C:\\Users\\vinayak\\Desktop\\Backtesting\\Plot.py'>

In [84]:
12*-153.84075609472146 + 19 * -253.15421107797258 

-6656.019083618137